# Amazon ML 2026 — Colab T4 / 12.7 GiB RAM

Run cells from top to bottom with **T4 GPU** selected. This version streams large allocations and keeps the original model, candidate budgets, features, folds, and 30-million training-pair cap.

The dataset path is filled in below. A real 3% training-slice comparison must pass before full mode starts. It verifies identical candidates/features and no measured validation-score decrease; it cannot guarantee the private leaderboard score.

Plan for **40 GiB free local capacity for the full job** and substantial Drive checkpoint space (roughly 80–100 GB recommended). The 40 GiB value is a planning guard, not a measured full-dataset peak. Completed stages are saved to Drive; interruption can repeat the unfinished stage. An exact country search index above 9 GiB stops with a clear error rather than reducing recall.

Use the new RUN_NAME below so your old code lock does not load the previous 28 GiB runner.


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

# Existing file: https://drive.google.com/file/d/1IDOadTPN0lM9XnuYjNyJuOwFJRIxFfYQ/view
DATASET_PATH = Path('/content/drive/MyDrive/6ab10eb3b23ba_student_resource.zip')
MODE = 'full'  # 'full': train + all test outputs; 'dev': 3% train check only
RUN_NAME = 'ber_lowram_v2'  # Keep this to resume. Change for a NEW dataset/settings run.
DENSE_BATCH = 512  # T4; lower to 256 after CUDA out-of-memory, then rerun.
DRIVE_RUN = Path('/content/drive/MyDrive/ML-2-C-26/runs') / RUN_NAME
LOCAL_RUN = Path('/content/ber_runs') / RUN_NAME
REPO_URL = 'https://github.com/SanskariXD/ML-2-C-26.git'
CODE_REF = 'main'  # First run resolves this once; the commit is then locked on Drive.

assert MODE in ('full', 'dev')
assert RUN_NAME and all(c.isalnum() or c in '-_' for c in RUN_NAME), 'Use letters/numbers/-/_ in RUN_NAME'
assert DATASET_PATH.is_file(), f'Drive file not found: {DATASET_PATH}. Mount the account containing the linked ZIP, or adjust its path.'
DRIVE_RUN.mkdir(parents=True, exist_ok=True)
LOCAL_RUN.mkdir(parents=True, exist_ok=True)
print('Dataset:', DATASET_PATH)
print('Outputs and completed-stage backups:', DRIVE_RUN)


## 1. Fetch the code and lock its version
A resumed run uses the same commit. A new RUN_NAME picks up newer code. The repository is public, so no GitHub token is required.


In [ ]:
import os, sys, subprocess, shutil
CODE = Path('/content/ML-2-C-26')
def checked(args, **kwargs):
    return subprocess.run([str(x) for x in args], check=True, **kwargs)
if not (CODE / '.git').exists():
    checked(['git', 'clone', REPO_URL, CODE])
assert subprocess.check_output(['git', '-C', str(CODE), 'remote', 'get-url', 'origin'], text=True).strip() == REPO_URL
assert not subprocess.check_output(['git', '-C', str(CODE), 'status', '--porcelain'], text=True).strip(), 'Local code was edited. Save those edits before switching versions.'
lock = DRIVE_RUN / 'code_commit.txt'
if lock.exists():
    COMMIT = lock.read_text().strip()
else:
    checked(['git', '-C', CODE, 'fetch', 'origin', CODE_REF])
    COMMIT = subprocess.check_output(['git', '-C', str(CODE), 'rev-parse', 'FETCH_HEAD'], text=True).strip()
    tmp = lock.with_suffix('.partial')
    tmp.write_text(COMMIT + '\n')
    os.replace(tmp, lock)
checked(['git', '-C', CODE, 'fetch', 'origin', COMMIT])
checked(['git', '-C', CODE, 'checkout', '--detach', COMMIT])
print('Locked commit:', COMMIT)

# Stream child output into the cell and repeat its final error for easy diagnosis.
def checked_live(args):
    from collections import deque
    import signal
    tail = deque(maxlen=60)
    with subprocess.Popen([str(x) for x in args], stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1,
                          start_new_session=True) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                tail.append(line)
            rc = process.wait()
        except KeyboardInterrupt:
            os.killpg(process.pid, signal.SIGTERM)
            raise
    if rc:
        print(''.join(tail), flush=True)
        raise RuntimeError(f'Command stopped with status {rc}; actual error is printed above.')


## 2. Install the isolated environment
The first run downloads Python, CUDA-compatible PyTorch and the pinned dependencies. PyTorch uses the runtime NVIDIA driver; the TPU is unsupported by this pipeline.


In [ ]:
checked([sys.executable, '-m', 'pip', 'install', '-q', 'uv==0.8.22'])
UV = shutil.which('uv')
assert UV, 'uv installation failed'
ENV = Path('/content/ber-py311')
PY = ENV / 'bin/python'
if not PY.exists():
    checked([UV, 'python', 'install', '3.11.13'])
    checked([UV, 'venv', '--python', '3.11.13', ENV])
BER = CODE / 'code/business_entity_resolution'
checked([UV, 'pip', 'install', '--python', PY, 'torch==2.7.1',
         '--index-url', 'https://download.pytorch.org/whl/cu126'])
checked([UV, 'pip', 'install', '--python', PY, '-r', BER / 'requirements.txt',
         '-r', BER / 'requirements-dense.txt'])
checked([UV, 'pip', 'check', '--python', PY])
checked([PY, '-c', "import torch, psutil, shutil, sentence_transformers, faiss; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'); print('RAM GiB:', round(psutil.virtual_memory().total/2**30,1)); print('Free disk GiB:', round(shutil.disk_usage('/content').free/2**30,1)); assert torch.cuda.is_available(), 'Choose T4 GPU in runtime settings'"])


## 3. Test installation and copy the existing dataset into runtime storage
The original Drive ZIP stays in place. Only the seven required TSV files are extracted, with checksums. This does not upload or publish your data. The synthetic smoke test checks the CPU pipeline; a small embedding check tests the actual GPU/model download.


In [ ]:
test_env = os.environ.copy()
for key in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS'):
    test_env[key] = '2'
checked([PY, BER / 'tests/smoke_test.py'], env=test_env)
checked([PY, '-c', "from sentence_transformers import SentenceTransformer; m=SentenceTransformer('intfloat/multilingual-e5-small', device='cuda'); x=m.encode(['query: sample business, Bengaluru'], normalize_embeddings=True); assert x.shape==(1,384); print('GPU embedding check PASS')"], env=test_env)
sys.path.insert(0, str(CODE / 'colab'))
from runner import stage_dataset
DATASET = LOCAL_RUN / 'dataset'
stage_dataset(DATASET_PATH, DATASET)
print('Dataset verified:', DATASET)


## 3b. Verify the memory-saving path on your data

This runs both execution paths on the same 3% training slice. E5 embeddings are computed once and shared. Candidate IDs, ordering, scores, and feature values must match; validation scores must not decrease. The check uses part of the data only for comparison. **The following full run still processes all train/test files.**

A matching saved report skips this check on resume. If it fails, share `low_memory_validation.json` (if present) and the error output. Do not bypass the check.


In [ ]:
checked_live([PY, '-u', CODE / 'colab/validate_low_memory.py', '--data', DATASET,
         '--work', LOCAL_RUN / 'validation',
         '--report', DRIVE_RUN / 'low_memory_validation.json', '--batch', DENSE_BATCH])


## 4. Train, predict, validate, and save to Drive
**This cell starts the full run after the comparison passes.** Logs stream below. Full mode runs six stages; after each, wait for `SAVED TO DRIVE`. Closing/resetting the runtime before that message can lose the current stage.

If interrupted: reconnect with a T4, keep the same RUN_NAME/MODE and rerun all cells. Complete checkpoints are restored automatically. Partial work from the failed stage is discarded. `dev` mode uses a 3% training subset and never generates a test submission; use a separate RUN_NAME for full mode.


In [ ]:
import psutil
print('System RAM GiB:', round(psutil.virtual_memory().total / 2**30, 1))
print('Free local disk GiB:', round(shutil.disk_usage('/content').free / 2**30, 1))
checked_live([PY, '-u', CODE / 'colab/runner.py', '--data', DATASET,
              '--local', LOCAL_RUN, '--drive', DRIVE_RUN,
              '--mode', MODE, '--batch', DENSE_BATCH, '--profile', 'lowram'])


## 5. Your files
Leaderboard upload: **matching_results.tsv**. Keep **candidate_pairs.tsv** and the code for the final package. The ZIP here contains output TSVs only; it is not the complete final documentation/code package.


In [ ]:
print('Saved in Google Drive:', DRIVE_RUN)
for name in ['output/matching_results.tsv', 'output/candidate_pairs.tsv', 'ber_submission.zip', 'report.json']:
    p = DRIVE_RUN / name
    if p.exists():
        print(name, f'{p.stat().st_size / 2**20:.1f} MiB')
print('For further details: https://github.com/SanskariXD/ML-2-C-26/blob/main/docs/COLAB_GUIDE.md')
